# Circulation Graph
Spatial connectivity through apertures (doors and windows).  
Two rooms are connected only when a door or window sits on their shared face.

## 1. Import Libraries

In [32]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper
from topologicpy.Color import Color

In [33]:
print(Helper.Version())

The version that you are using (0.9.20) is OLDER than the latest version (0.9.22) from PyPI. Please consider upgrading to the latest version.


In [34]:
renderer = "vscode"

## 2. Import OBJ Files

In [35]:
objects = Topology.ByOBJPath(r"C:\Users\danie\Documents\GitHub\term 3\graph ml\assignment01\Geometry\base model.obj", selfMerge=True)
print("Objects:", objects)

Objects: [<topologic_core.Cluster object at 0x0000021878534AB0>]


In [36]:
doors   = Topology.ByOBJPath(r"C:\Users\danie\Documents\GitHub\term 3\graph ml\assignment01\Geometry\door.obj",   selfMerge=True)
windows = Topology.ByOBJPath(r"C:\Users\danie\Documents\GitHub\term 3\graph ml\assignment01\Geometry\window.obj", selfMerge=True)

# Flatten clusters to individual faces and colour-code
aperture_faces = []
for ap in doors:
    for f in (Topology.Faces(ap) or [ap]):
        Topology.SetDictionary(f, Dictionary.ByKeysValues(["color"], ["purple"]))
        aperture_faces.append(f)
for ap in windows:
    for f in (Topology.Faces(ap) or [ap]):
        Topology.SetDictionary(f, Dictionary.ByKeysValues(["color"], ["pink"]))
        aperture_faces.append(f)

print("Aperture faces:", len(aperture_faces))

Aperture faces: 22


## 3. Extract Cells and Build CellComplex

In [37]:
cells = Topology.Cells(objects[0])
print("Number of cells:", len(cells))

cc = CellComplex.ByCells(cells)
cc = Topology.RemoveCoplanarFaces(cc)
cc = Topology.RemoveCollinearEdges(cc)
print("CellComplex:", cc)

Number of cells: 18
CellComplex: <topologic_core.CellComplex object at 0x0000021877C41C30>


## 4. Show Geometry

In [45]:
ap_cluster = Cluster.ByTopologies(aperture_faces)
Topology.Show(cc, ap_cluster,
              faceColorKey="color",
              faceOpacity=0.5,
              backgroundColor="white",
              width=700,
              height=500,
              renderer=renderer)

## 5. Add Apertures to CellComplex

In [39]:
cc = Topology.AddApertures(cc, aperture_faces, subTopologyType="face")
print("Apertures added:", len(aperture_faces))

Apertures added: 22


## 6. Build the Circulation Graph
`useApertures=True` — cells connect only through a door or window on their shared face.

In [40]:
g_circ = Graph.ByTopology(cc,
                           direct=False,
                           viaSharedApertures=True,
                           toExteriorApertures=False)
print("Vertices:", len(Graph.Vertices(g_circ)))
print("Edges:",    len(Graph.Edges(g_circ)))

Vertices: 30
Edges: 24


## 7. Assign Visual Attributes

In [41]:
for v in Graph.Vertices(g_circ):
    d = Dictionary.ByKeysValues(["size", "color"], [18, "red"])
    Topology.SetDictionary(v, d)

for e in Graph.Edges(g_circ):
    d = Dictionary.ByKeysValues(["width", "color"], [4, "black"])
    Topology.SetDictionary(e, d)

## 8. Show Circulation Graph

In [42]:
Topology.Show(g_circ,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              backgroundColor="white",
              width=700,
              height=500,
              renderer=renderer)

## 9. Show Circulation Graph Overlaid on Geometry

In [43]:
ap_cluster = Cluster.ByTopologies(aperture_faces)
Topology.Show(cc, ap_cluster, g_circ,
              faceColorKey="color",
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceOpacity=0.4,
              backgroundColor="white",
              width=700,
              height=500,
              renderer=renderer)